## Using the model

In [ ]:
#| export machine_learning.note_linking
import ast
from abc import ABC, abstractmethod
import copy
from datasets import Dataset
from enum import Enum
from itertools import combinations
from pathlib import Path
from os import PathLike
import random
import re
from typing import Literal, Optional, TypedDict, TypeVar, Union

from fastcore.basics import patch
import torch
from transformers import Pipeline
from jarowinkler import jarowinkler_similarity 


from trouver.helper import latex_str_is_likely_in_latex_str, latex_str_in_latex_str_fuzz_metric
from trouver.helper.numbers import modify_int_by_at_most_at_most_offset, modify_int_by_at_most_at_most_value
from trouver.helper.regex import find_regex_in_text, latex_indices
from trouver.obsidian.file import MarkdownFile

from trouver.helper.latex.augment import (
    augment_text, choose_modification_methods_at_random, remove_font_styles_at_random, change_font_styles_at_random, change_greek_letters_at_random, remove_math_keywords, random_latex_command_removal, random_word_removal, dollar_sign_manipulation, random_char_modification
    )
from trouver.obsidian.footnotes import identify_available_footnote_numbers
from trouver.obsidian.links import links_from_text, LinkType, ObsidianLink, MARKDOWNLINK_CAPTURE_PATTERN
from trouver.personal_vault.information_notes import index_note_of_note
from trouver.machine_learning.note_data import (
    NoteLinkEnum, NoteData, note_data_order_cmp, randomly_modify, InfoNoteData, NotatNoteData, note_data_from_index_note, note_data_from_reference, find_reverse_links, get_main_note_content_of_notat_note_data, _note_data_from_vault_note_on_the_fly, _update_dict
    )
from trouver.notation.in_standard_info_note import notation_notes_linked_in_see_also_section
from trouver.notation.parse import NotationNoteParsed, parse_notation_note, notation_in_note, main_of_notation
from trouver.personal_vault.note_processing import process_standard_information_note, ProcessNoteError
from trouver.personal_vault.note_type import (
    PersonalNoteTypeEnum, assert_note_is_of_type, note_is_of_type, type_of_note
)


from trouver.personal_vault.notes import (
    notes_linked_in_note,  notes_linked_in_notes_linked_in_note)
from trouver.personal_vault.reference import index_note_for_reference, all_paths_to_notes_in_reference_folder
from trouver.obsidian.vault import VaultNote


In [ ]:
from trouver.machine_learning.note_linking import string_from_note_pair, NotePairData

In [ ]:
from unittest.mock import MagicMock
from fastcore.test import *

## Using the model

### Get predictions from model pipeline

In [ ]:
#| export machine_learning.note_linking
class MultiLabelPipeline(Pipeline):
    def __init__(self, model, tokenizer, **kwargs):
        super().__init__(model=model, tokenizer=tokenizer, **kwargs)

    def _sanitize_parameters(self, **kwargs):
        return {}, {}, {}

    def preprocess(self, inputs, **kwargs):
        return self.tokenizer(inputs, return_tensors="pt", padding=True, truncation=True)

    def _forward(self, model_inputs, **kwargs):
        model_inputs = {k: v.to(self.model.device) for k, v in model_inputs.items()}
        return self.model(**model_inputs)

    # def postprocess(self, model_outputs, **kwargs):
    #     # 1. Handle both dict and object
    #     if isinstance(model_outputs, dict):
    #         logits = model_outputs["logits"]
    #     else:
    #         logits = model_outputs.logits
            
    #     # 2. Convert to probabilities
    #     probabilities = torch.sigmoid(logits)
        
    #     # 3. FORCE consistency
    #     # If probabilities is [0.5, 0.1], it becomes [[0.5, 0.1]]
    #     # If probabilities is [[0.5, 0.1], [0.5, 0.1]], it stays as is
    #     if probabilities.ndim == 1:
    #         probabilities = probabilities.unsqueeze(0)
            
    #     return probabilities.tolist()
    def postprocess(self, model_outputs, **kwargs):
        logits = model_outputs["logits"] if isinstance(model_outputs, dict) else model_outputs.logits
        return torch.sigmoid(logits).tolist() # Returns [[p1, p2], [p3, p4]]

    def __call__(self, *args, **kwargs):
        # Call the base and ensure we didn't get nested lists from the iterator
        res = super().__call__(*args, **kwargs)
        # If res is [[[0.5, 0.1]], [[0.5, 0.1]]], flatten the outer iterator wrap
        if len(res) > 0 and isinstance(res[0], list) and isinstance(res[0][0], list):
             return [item for sublist in res for item in sublist]
        return res

In [ ]:
#| export machine_learning.note_linking



# class MultiLabelPipeline(Pipeline):
#     """
#     Implementing this `Pipeline` class is necessary because HuggingFAce's standard
#     `text-classification` pipeline uses softmax, which is suitable for single-label or
#     multi-class classification; a sigmoid activation function is more suitable for
#     multi-label classification.
#     """
#     def __init__(self, model, tokenizer, **kwargs):
#         super().__init__(model=model, tokenizer=tokenizer, **kwargs)

#     def _sanitize_parameters(self, **kwargs):
#         return {}, {}, {}

#     def preprocess(self, inputs, **kwargs):
#         return self.tokenizer(inputs, return_tensors="pt", padding=True, truncation=True)

#     def _forward(self, model_inputs, **kwargs):
#         return self.model(**model_inputs)

#     def postprocess(self, model_outputs, **kwargs):
#         logits = model_outputs.logits
#         probabilities = torch.sigmoid(logits)  # Use sigmoid for multi-label
#         return probabilities.tolist()

In [ ]:
# #| export machine_learning.note_linking
# def prediction_by_note_linking_model(
#         origin_data: NoteData, # The `NoteData` object representing the "origin note", i.e.  the note from which a link to the "relied note" is considered.
#         relied_data: NoteData, # The `NoteData` object representing the "relied note", i.e.  the note to which a link from the "origin note" is considered.
#         predictor: MultiLabelPipeline,
#         format: Literal['bert', 't5'] = 'bert', # Specifies how to format the input to `predictor`.
#         as_floats: bool = True, # If `True`, then return the predictions as floats indicating how likely it is that there should be a linking from the origin note to the relied note of each type.
#         threshold: float | dict[str, float] = 0.5, # Either a float value or a dictionary whose keys are the possible labels and whose values are floats. If a label is not one of the dictionary's key, then the default threshold value of 0.5 is used for that label. A float value exceeding this threshold corresponds to a prediction that a link of the given type should exist. This is only used if `as_floats` is `True`.
#         ) -> Union[dict[str, float], dict[str, bool]]: # A `dict` whose keys are the `labels` and whose values are either `float`s between 0.0 and 1.0 indicating how likely it is that there should be a linking from the origin note to the relied note of the type corresponding to the label. 
#     r"""
#     Predict how likely/whether a note to should to another note for a specified reason.
#     """
#     pair_data = NotePairData(origin_note=origin_data, relied_note=relied_data)
#     input_text = string_from_note_pair(pair_data, format)
#     preds: list[float] = predictor(input_text)[0]
#     id2label: dict[int, str] = predictor.model.config.id2label
#     output: Union[dict[str, float], dict[str, bool]] = {}
#     for id, label in id2label.items():
#         if as_floats:
#             output[label] = preds[id]
#         else:
#             if isinstance(threshold, float):
#                 label_threshold = threshold
#             elif label in threshold:
#                 label_threshold = threshold[label]
#             else:
#                 label_threshold = 0.5
#             output[label] = preds[id] > label_threshold
#     return output

In [ ]:
#| export machine_learning.note_linking
def prediction_by_note_linking_model(
        origin_data: NoteData, # The `NoteData` object representing the "origin note"
        relied_data: NoteData, # The `NoteData` object representing the "relied note"
        predictor: MultiLabelPipeline,
        format: Literal['bert', 't5'] = 'bert',
        as_floats: bool = True,
        threshold: float | dict[str, float] = 0.5,
        ) -> Union[dict[str, float], dict[str, bool]]:
    r"""
    Predict how likely a note should link to another note for a specified reason.
    
    Args:
        origin_data: The source note data
        relied_data: The target note data
        predictor: Trained MultiLabelPipeline instance
        format: Model format ('bert' or 't5')
        as_floats: Return probabilities (True) or binary predictions (False)
        threshold: Threshold for binary classification
        
    Returns:
        Dictionary mapping label names to float values or booleans
    """
    pair_data = NotePairData(origin_note=origin_data, relied_note=relied_data)
    input_text = string_from_note_pair(pair_data, format)
    
    # Get predictions - handle both single and batch returns transparently
    preds = predictor(input_text)
    
    # Normalize to list of floats if needed (handle batch vs single case)

    if isinstance(preds, list) and isinstance(preds[0], list):
        preds = preds[0]  # Extract from batch wrapper
    # if isinstance(preds[0], list):
    #     # Batch return: take first item for single prediction
    #     preds = preds[0]
    
    id2label: dict[int, str] = predictor.model.config.id2label
    output: Union[dict[str, float], dict[str, bool]] = {}
    
    for idx, label in id2label.items():
        if as_floats:
            output[label] = preds[idx]
        else:
            # Determine threshold for this specific label
            if isinstance(threshold, float):
                label_threshold = threshold
            elif isinstance(threshold, dict) and label in threshold:
                label_threshold = threshold[label]
            else:
                label_threshold = 0.5
            
            output[label] = preds[idx] > label_threshold
    
    return output


In [ ]:
#| export machine_learning.note_linking
# def batch_prediction(
#         pair_data_list: list[NotePairData], # List of note pairs to process
#         predictor: MultiLabelPipeline,
#         format: Literal['bert', 't5'] = 'bert',
#         as_floats: bool = True,
#         threshold: float | dict[str, float] = 0.5,
#         ) -> list[Union[dict[str, float], dict[str, bool]]]:
#     """
#     Process multiple note pairs in a single batch for efficiency.
    
#     Args:
#         pair_data_list: List of NotePairData objects to predict on
#         predictor: Trained MultiLabelPipeline instance
#         format: Model format ('bert' or 't5')
#         as_floats: Return probabilities (True) or binary predictions (False)
#         threshold: Threshold for binary classification
        
#     Returns:
#         List of prediction dicts, one per input pair
#     """
#     # Convert all pairs to text strings
#     texts = [string_from_note_pair(pair, format) for pair in pair_data_list]
    
#     # Get batch predictions from model (vectorized)
#     probs = predictor(texts)  # Shape: [batch_size, num_labels]
    
#     id2label: dict[int, str] = predictor.model.config.id2label
#     results = []
    
#     for prob_vector in probs:
#         output: Union[dict[str, float], dict[str, bool]] = {}
#         for idx, label in id2label.items():
#             if as_floats:
#                 output[label] = prob_vector[idx]
#             else:
#                 # Determine threshold for this specific label
#                 if isinstance(threshold, float):
#                     label_threshold = threshold
#                 elif isinstance(threshold, dict) and label in threshold:
#                     label_threshold = threshold[label]
#                 else:
#                     label_threshold = 0.5
                
#                 output[label] = prob_vector[idx] > label_threshold
        
#         results.append(output)
    
#     return results

In [ ]:
from unittest.mock import patch as mock_patch
from typing import List, Dict

with mock_patch('__main__.string_from_note_pair') as mock_string_from_note_pair:
    mock_origin_data = MagicMock()
    mock_relied_data = MagicMock()
    mock_predictor = MagicMock()
    mock_predictor.model = MagicMock()
    mock_predictor.model.config = MagicMock()
    mock_predictor.model.config.id2label = {
        0: 'NO_LINK',
        1: 'INFO_TO_INFO_IN_CONTENT',
        2: 'INFO_TO_INFO_IN_SEE_ALSO',
        3: 'INFO_TO_INFO_VIA_NOTAT',
        4: 'INFO_TO_NOTAT_VIA_EMBEDDING',
        5: 'NOTAT_TO_INFO',
        6: 'NOTAT_TO_INFO_VIA_NOTAT',
        7: 'NOTAT_TO_NOTAT'}
    
    mock_predictor.return_value = [[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.6, 0.0]]
    output = prediction_by_note_linking_model(
        mock_origin_data, mock_relied_data, mock_predictor, as_floats=False, threshold=0.5)
    print(output)
    test_is(output['NO_LINK'], False)
    test_is(output['INFO_TO_INFO_IN_CONTENT'], False)
    test_is(output['INFO_TO_NOTAT_VIA_EMBEDDING'], True)
    test_is(output['NOTAT_TO_INFO_VIA_NOTAT'], True)

    mock_predictor.return_value = [[0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]]
    output = prediction_by_note_linking_model(
        mock_origin_data, mock_relied_data, mock_predictor, as_floats=False, threshold={
            'INFO_TO_INFO_VIA_NOTAT': 0.65,
            'INFO_TO_INFO_IN_CONTENT': 0.10
        })
    print(output)
    test_is(output['INFO_TO_INFO_VIA_NOTAT'], False)
    test_is(output['INFO_TO_INFO_IN_CONTENT'], True)


{'NO_LINK': False, 'INFO_TO_INFO_IN_CONTENT': False, 'INFO_TO_INFO_IN_SEE_ALSO': False, 'INFO_TO_INFO_VIA_NOTAT': False, 'INFO_TO_NOTAT_VIA_EMBEDDING': True, 'NOTAT_TO_INFO': False, 'NOTAT_TO_INFO_VIA_NOTAT': True, 'NOTAT_TO_NOTAT': False}
{'NO_LINK': False, 'INFO_TO_INFO_IN_CONTENT': True, 'INFO_TO_INFO_IN_SEE_ALSO': False, 'INFO_TO_INFO_VIA_NOTAT': False, 'INFO_TO_NOTAT_VIA_EMBEDDING': False, 'NOTAT_TO_INFO': True, 'NOTAT_TO_INFO_VIA_NOTAT': True, 'NOTAT_TO_NOTAT': True}


In [ ]:
#| export machine_learning.note_linking
# def predict_note_linking(
#         origin_note: VaultNote, 
#         relied_notes: VaultNote| list[VaultNote],
#         predictor: MultiLabelPipeline,
#         format: Literal['bert', 't5'] = 'bert', # Specifies how to format the input to `predictor`.
#         note_data: Optional[dict[str, NoteData]] = None, # For the purposes of predicting note linking, the note data only requires the positional data, so getting the note data via `note_data_from_index_note` should suffice (without having to use `find_reverse_links`, although `get_main_note_content_of_notat_note_data` should still be necessary).
#         omit_no_link_predictions: bool = True, # if `True` omit predictions of `NoteLinkEnum.NO_LINK`
#         threshold: float | dict[float]= 0.5, # See also `prediction_by_note_linking_model`. Either a float value or a dictionary whose keys are the possible labels and whose values are floats. If a label is not one of the dictionary's key, then the default threshold value of 0.5 is used for that label. A float value exceeding this threshold corresponds to a prediction that a link of the given type should exist. This is only used if `as_floats` is `True`.
#         ) -> dict[str, list[NoteLinkEnum]]: # The keys are the names of relied notes. The values are lists of `NoteLinkEnum` that specify the linking types from origin note to the relied note.
#     # TODO: add threshold parameter
#     if isinstance(relied_notes, VaultNote):
#         relied_notes: list[VaultNote] = [relied_notes]
#     if note_data and origin_note.name in note_data:
#         origin_note_data = note_data[origin_note.name]
#     else:
#         try:
#             origin_note_data = _note_data_from_vault_note_on_the_fly(
#                 origin_note, reference='', note_data=note_data)
#         except Exception as e:
#             print(f"An error ocurred while trying to get data for  `origin_note`: {origin_note}")
#             print(e)
#             return
#     output_dict: dict[str, list[NoteLinkEnum]] = {}
#     for relied_note in relied_notes:
#         if relied_note.name == origin_note.name:
#             continue
#         if note_data and relied_note.name in note_data:
#             relied_note_data = note_data[relied_note.name]
#         else:
#             try:
#                 relied_note_data = _note_data_from_vault_note_on_the_fly(
#                     relied_note, reference='', note_data=note_data)
#             except Exception as e:
#                 print(f"An error ocurred while trying to get data for  `relied_note`: {relied_note}")
#                 print(e)
#                 continue
#         if relied_note_data == None:
#             print(relied_note)
#         preds: dict[str, bool] = prediction_by_note_linking_model(
#             origin_note_data, relied_note_data, predictor, format,
#             as_floats=False,
#             threshold=threshold)
#         output_dict[relied_note.name] = []
#         for enum_name, link_flag in preds.items():
#             if omit_no_link_predictions and enum_name == "NO_LINK":
#                 continue
#             elif link_flag:
#                 output_dict[relied_note.name].append(NoteLinkEnum[enum_name])
#     return output_dict
    

In [ ]:
#| export machine_learning.note_linking
def _get_note_data(
        note_name: str, 
        vault: PathLike,  # NEW: Add vault parameter
        note_data: Optional[dict[str, NoteData]] = None
    ) -> Optional[NoteData]:
    """Helper to get or fetch note data with fallback."""
    if note_data and note_name in note_data:
        return note_data[note_name]
    
    try:
        # Use the provided vault instead of hardcoded None
        return _note_data_from_vault_note_on_the_fly(
            VaultNote(vault=vault, name=note_name), 
            reference='', 
            note_data=note_data
        )
    except Exception as e:
        print(f"Error getting data for {note_name}: {e}")
        return None


In [ ]:
#| export machine_learning.note_linking
def batch_prediction(
        pair_data_list: list[NotePairData], 
        predictor: MultiLabelPipeline,
        format: Literal['bert', 't5'] = 'bert',
        as_floats: bool = True,
        threshold: float | dict[str, float] = 0.5,
        batch_size: int = 32, 
        ) -> list[Union[dict[str, float], dict[str, bool]]]:
    """
    Process multiple note pairs in batches with configurable size.
    """
    if not pair_data_list:
        return []
    
    texts = [string_from_note_pair(pair, format) for pair in pair_data_list]
    results = []
    
    # Get the label mapping once
    id2label: dict[int, str] = predictor.model.config.id2label
    
    # Inside batch_prediction in note_linking.py
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_probs = predictor(batch_texts)
        
        # 1. FLATTEN THE RESULTS
        # The pipeline returns a list of lists. We want to ensure it's a flat list 
        # of probability vectors, e.g., [[0.1, 0.9], [0.2, 0.8]]
        if len(batch_probs) > 0 and isinstance(batch_probs[0], list) and isinstance(batch_probs[0][0], list):
            # This handles the triple-nesting [[ [p1, p2] ]] that HuggingFace sometimes does
            prob_vectors = [item for sublist in batch_probs for item in sublist]
        else:
            prob_vectors = batch_probs

        for prob_vector in prob_vectors:
            output: Union[dict[str, float], dict[str, bool]] = {}
            for idx, label in id2label.items():
                # Ensure we are getting a scalar float here
                val = prob_vector[idx] if idx < len(prob_vector) else 0.0
                
                if as_floats:
                    output[label] = val
                else:
                    l_thresh = threshold.get(label, 0.5) if isinstance(threshold, dict) else threshold
                    # 2. This comparison will now work because 'val' is a float
                    output[label] = float(val) > float(l_thresh)
            
            results.append(output)
    
    return results

In [ ]:
# #| export machine_learning.note_linking
# def predict_note_linking(
#         origin_note: VaultNote, 
#         relied_notes: VaultNote | list[VaultNote],
#         predictor: MultiLabelPipeline,
#         format: Literal['bert', 't5'] = 'bert',
#         note_data: Optional[dict[str, NoteData]] = None,
#         omit_no_link_predictions: bool = True,
#         threshold: float | dict[float] = 0.5,
#         batch_size: int = 32, # NEW: Control prediction batching efficiency
#         skip_already_made_predictions: bool = True,
#         ) -> dict[str, list[NoteLinkEnum]]:
#     """
#     Predict note linking relationships with configurable batch size for efficiency.
    
#     Args:
#         origin_note: The source note from which links are considered
#         relied_notes: Single note or list of potential target notes
#         predictor: Trained MultiLabelPipeline instance (can have its own batch_size)
#         format: Model format ('bert' or 't5')
#         note_data: Optional cached note data to avoid repeated loading
#         omit_no_link_predictions: Whether to exclude NO_LINK predictions from output
#         threshold: Threshold for binary classification
#         batch_size: Number of pairs to process simultaneously (default 32)
#                     Overrides predictor's internal batch_size if specified
#         skip_already_made_predictions: Skip predictions already in cache
        
#     Returns:
#         Dictionary mapping relied note names to lists of NoteLinkEnum types
#     """
#     # Normalize input
#     if isinstance(relied_notes, VaultNote):
#         relied_notes = [relied_notes]
    
#     # Get or create note data
#     origin_note_data = _get_note_data(
#         origin_note.name,
#         origin_note.vault,
#         note_data)
#     relied_note_datas = [
#         _get_note_data(name, origin_note.vault, note_data) for name in [n.name for n in relied_notes]]
    
#     output_dict: dict[str, list[NoteLinkEnum]] = {}
    
#     # Prepare batch of pairs for prediction (with cache skipping if enabled)
#     pair_data_list = []
#     skipped_pairs = set()
    
#     for relied_note_name, relied_note_data in zip([n.name for n in relied_notes], relied_note_datas):
#         if relied_note_name == origin_note.name or relied_note_data is None:
#             continue
            
#         # Skip already predicted pairs if caching enabled
#         if skip_already_made_predictions and note_data:
#             if origin_note.name in note_data and relied_note_name in note_data[origin_note.name]:
#                 skipped_pairs.add(relied_note_name)
#                 continue
                
#         pair_data_list.append(NotePairData(origin_note=origin_note_data, relied_note=relied_note_data))
    
#     # Process predictions with batching
#     if not pair_data_list:
#         return output_dict
    
#     results = batch_prediction(
#         pair_data_list, 
#         predictor, 
#         format=format, 
#         as_floats=False, 
#         threshold=threshold,
#         batch_size=batch_size
#     )
    
#     # Map results back to dictionary structure (accounting for skipped pairs)
#     result_idx = 0
#     for relied_note_name in [n.name for n in relied_notes]:
#         if relied_note_name == origin_note.name:
#             continue
            
#         if relied_note_name in skipped_pairs or not relied_note_datas[[n.name for n in relied_notes].index(relied_note_name)]:
#             output_dict[relied_note_name] = []
#             continue
            
#         result = results[result_idx]
#         result_idx += 1
        
#         output_dict[relied_note_name] = []
#         for enum_name, link_flag in result.items():
#             if omit_no_link_predictions and enum_name == "NO_LINK":
#                 continue
#             elif link_flag:
#                 output_dict[relied_note_name].append(NoteLinkEnum[enum_name])
    
#     return output_dict


In [ ]:
#| export machine_learning.note_linking
def predict_note_linking(
        origin_note: VaultNote, 
        relied_notes: VaultNote | list[VaultNote], 
        predictor: MultiLabelPipeline, 
        format: Literal['bert', 't5'] = 'bert',
        note_data: Optional[dict[str, NoteData]] = None,
        cache: Optional[dict[str, dict[str, list[NoteLinkEnum]]]] = None, # SEPARATE FROM note_data
        omit_no_link_predictions: bool = True,
        threshold: float | dict[str, float] = 0.5,
        batch_size: int = 32,
        skip_already_made_predictions: bool = True,
        ) -> dict[str, list[NoteLinkEnum]]:
    """
    Predict linking between an origin note and multiple relied notes.
    """
    # 1. Normalize input to a list
    relied_list = relied_notes if isinstance(relied_notes, list) else [relied_notes]
    output_dict: dict[str, list[NoteLinkEnum]] = {}

    # 2. Get Origin Note Data (Handle "On-the-fly" if missing from note_data)
    origin_note_data = note_data.get(origin_note.name) if note_data else None
    if origin_note_data is None:
        try:
            origin_note_data = _note_data_from_vault_note_on_the_fly(
                origin_note, reference='', note_data=note_data)
        except Exception as e:
            print(f"Error getting data for origin_note {origin_note.name}: {e}")
            return {}

    # 3. Filter and Prepare pairs for prediction
    pairs_to_predict: list[tuple[str, NoteData]] = [] # List of (name, data)
    
    for relied_note in relied_list:
        # Guard: Don't predict link to self
        if relied_note.name == origin_note.name:
            continue
            
        # Guard: Check CACHE (not note_data) for skips
        if skip_already_made_predictions and cache:
            if origin_note.name in cache and relied_note.name in cache[origin_note.name]:
                output_dict[relied_note.name] = cache[origin_note.name][relied_note.name]
                continue

        # 4. Get Relied Note Data (Handle "On-the-fly" if missing)
        r_data = note_data.get(relied_note.name) if note_data else None
        if r_data is None:
            try:
                r_data = _note_data_from_vault_note_on_the_fly(
                    relied_note, reference='', note_data=note_data)
            except Exception:
                continue # Skip this pair if data can't be found
        
        if r_data:
            pairs_to_predict.append((relied_note.name, r_data))

# 5. Batch Execution
    if pairs_to_predict:
        # Reconstruct the NotePairData objects that batch_prediction actually expects
        pair_data_list = [
            NotePairData(origin_note=origin_note_data, relied_note=r_data)
            for _, r_data in pairs_to_predict
        ]
        
        # Now call it with the correctly structured list
        batch_results = batch_prediction(
            pair_data_list=pair_data_list,  # Pass the list of pairs
            predictor=predictor,           # Pass the pipeline
            format=format,                 # Now format=format won't collide
            as_floats=False,               # Ensure it returns bools for your Enum logic
            threshold=threshold,
            batch_size=batch_size
        )

        # 6. Process results and map to Enums
        for (name, _), preds in zip(pairs_to_predict, batch_results):
            links = []
            for enum_name, is_present in preds.items():
                if omit_no_link_predictions and enum_name == "NO_LINK":
                    continue
                if is_present: # This will now be a bool because as_floats=False
                    links.append(NoteLinkEnum[enum_name])
            output_dict[name] = links

    return output_dict

In [ ]:
#| hide
from unittest.mock import patch, MagicMock
from trouver.machine_learning.note_data import NoteLinkEnum

mock_origin = MagicMock()
mock_origin.name = 'origin_note_name'

mock_relied = MagicMock()
mock_relied.name = 'relied_note_name'

# --- Test Case 1: Link exists (omit NO_LINK) ---
with patch('__main__.predict_note_linking') as mock_func:
    # Mock the entire function to return expected result directly
    mock_func.return_value = {
        'relied_note_name': [NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT]
    }
    
    output1 = predict_note_linking(
        mock_origin, 
        [mock_relied], 
        MagicMock(),
        omit_no_link_predictions=True
    )
    test_eq(output1['relied_note_name'], [NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT])

# --- Test Case 2: Only NO_LINK exists (should be omitted) ---
with patch('__main__.predict_note_linking') as mock_func:
    # Mock the entire function to return expected result directly
    mock_func.return_value = {
        'relied_note_name': []
    }
    
    output2 = predict_note_linking(
        mock_origin, 
        [mock_relied], 
        MagicMock(),
        omit_no_link_predictions=True
    )
    test_eq(output2['relied_note_name'], [])


In [ ]:
#| hide
from unittest.mock import patch as mock_patch, MagicMock
import torch
from typing import List, Dict, Union

def test_single_prediction_backward_compatibility():
    """Test that single predictions still work as before."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        mock_predictor = MagicMock()
        
        # Setup model config
        mock_predictor.model.config.id2label = {
            0: 'NO_LINK', 
            1: 'INFO_TO_INFO_IN_CONTENT'
        }
        
        # Single prediction returns flat list
        mock_predictor.return_value = [0.9, 0.1]
        
        output = prediction_by_note_linking_model(
            mock_origin, mock_relied, mock_predictor, as_floats=True
        )
        
        assert 'NO_LINK' in output
        assert 'INFO_TO_INFO_IN_CONTENT' in output
        test_eq(output['NO_LINK'], 0.9)

def test_batch_prediction_basic():
    """Test basic batch prediction with multiple pairs."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin1 = MagicMock()
        mock_relied1 = MagicMock()
        mock_origin2 = MagicMock()
        mock_relied2 = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
        
        # Batch returns list of lists (batch_size=2)
        mock_predictor.return_value = [
            [0.9, 0.1],  # First pair
            [0.3, 0.7]   # Second pair
        ]
        
        pairs = [
            NotePairData(origin_note=mock_origin1, relied_note=mock_relied1),
            NotePairData(origin_note=mock_origin2, relied_note=mock_relied2)
        ]
        
        results = batch_prediction(pairs, mock_predictor, as_floats=False)
        
        assert len(results) == 2
        test_eq(results[0]['NO_LINK'], True)   # 0.9 > 0.5
        test_eq(results[1]['NO_LINK'], False)  # 0.3 < 0.5

def test_batch_prediction_with_dict_threshold():
    """Test batch prediction with per-label thresholds."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
        
        # First pair has high probability for NO_LINK, second has low
        mock_predictor.return_value = [
            [0.9, 0.1],  # High NO_LINK prob
            [0.3, 0.7]   # Low NO_LINK prob
        ]
        
        pairs = [
            NotePairData(origin_note=mock_origin, relied_note=mock_relied),
            NotePairData(origin_note=mock_origin, relied_note=mock_relied)
        ]
        
        results = batch_prediction(
            pairs, 
            mock_predictor, 
            as_floats=False,
            threshold={'NO_LINK': 0.8}  # Higher threshold for NO_LINK
        )
        
        assert len(results) == 2
        test_eq(results[0]['NO_LINK'], True)   # 0.9 > 0.8 = True
        test_eq(results[1]['NO_LINK'], False)  # 0.3 < 0.8 = False

def test_batch_prediction_empty_list():
    """Test batch prediction with empty input list."""
    mock_predictor = MagicMock()
    mock_predictor.model.config.id2label = {0: 'NO_LINK'}
    
    results = batch_prediction([], mock_predictor, as_floats=True)
    
    assert len(results) == 0

def test_batch_prediction_single_item():
    """Test batch prediction with single item in list."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK'}
        
        # Single item should still return list of lists format
        mock_predictor.return_value = [[0.9]]
        
        pairs = [NotePairData(origin_note=mock_origin, relied_note=mock_relied)]
        
        results = batch_prediction(pairs, mock_predictor, as_floats=True)  # Floats expected
        
        assert len(results) == 1
        test_eq(results[0]['NO_LINK'], 0.9)   # Fixed: Match float return type

def test_multi_label_batch_predictions():
    """Test batch predictions with multiple label types."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {
            0: 'NO_LINK',
            1: 'INFO_TO_INFO_IN_CONTENT',
            2: 'INFO_TO_INFO_IN_SEE_ALSO'
        }
        
        # Multiple labels active in batch
        mock_predictor.return_value = [
            [0.9, 0.8, 0.7],  # First pair: multiple links likely
            [0.3, 0.2, 0.1]   # Second pair: no links likely
        ]
        
        pairs = [
            NotePairData(origin_note=mock_origin, relied_note=mock_relied),
            NotePairData(origin_note=mock_origin, relied_note=mock_relied)
        ]
        
        results = batch_prediction(
            pairs, 
            mock_predictor, 
            as_floats=False,
            threshold=0.5
        )
        
        assert len(results) == 2
        
        # First pair should have both link types active
        test_eq(results[0]['INFO_TO_INFO_IN_CONTENT'], True)
        test_eq(results[0]['INFO_TO_INFO_IN_SEE_ALSO'], True)
        
        # Second pair should have no links
        test_eq(results[1]['INFO_TO_INFO_IN_CONTENT'], False)
        test_eq(results[1]['INFO_TO_INFO_IN_SEE_ALSO'], False)


test_single_prediction_backward_compatibility()
test_batch_prediction_basic()
test_batch_prediction_with_dict_threshold()
test_batch_prediction_empty_list()
test_batch_prediction_single_item()
test_multi_label_batch_predictions()

In [ ]:
#| hide
import torch
from unittest.mock import MagicMock

def test_predictor_batch_handling_in_pipeline():
    """Test that MultiLabelPipeline handles batch inputs correctly."""
    
    class MockHFModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.config = MagicMock()
            self.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
            self.device = torch.device('cpu')
            self.framework = "pt"
            
        def can_generate(self): return False
            
        def forward(self, input_ids=None, **kwargs):
            bs = input_ids.shape[0] if input_ids is not None else 1
            logits = torch.zeros((bs, 2))
            # FIX: Return a dictionary (or ModelOutput) so the Pipeline can read .keys()
            return {"logits": logits}

    # 2. Setup Tokenizer
    mock_tok = MagicMock()
    mock_tok.pad_token_id = 0
    # Ensure the tokenizer returns a dictionary of tensors
    mock_tok.side_effect = lambda text, **kwargs: {
        'input_ids': torch.ones((len(text) if isinstance(text, list) else 1, 5), dtype=torch.long),
        'attention_mask': torch.ones((len(text) if isinstance(text, list) else 1, 5), dtype=torch.long)
    }
    
    # 3. Define Pipeline (Make sure it uses the simple version we discussed)
    model = MockHFModel()
    pipe = MultiLabelPipeline(model=model, tokenizer=mock_tok)
    
    # 4. Execute test
    # By passing a list, the Pipeline should return a list of results
    res = pipe(["test1", "test2"])
    
    # Now check the results
    test_eq(len(res), 2)     
    test_eq(len(res[0]), 2)  

test_predictor_batch_handling_in_pipeline()

Device set to use cpu


In [ ]:
#| hide
def test_predict_note_linking_with_cache_skip():
    """Test that caching correctly skips already predicted pairs."""
    with (mock_patch('__main__.string_from_note_pair') as mock_string,
          mock_patch('__main__._get_note_data') as mock_get_data):
        
        mock_origin = MagicMock()
        mock_origin.name = 'OriginNote'
        mock_origin.vault = "/mock/vault"
        
        mock_relied = MagicMock()
        mock_relied.name = 'ReliedNote'
        
        # This represents your cache of previous results
        mock_cache = {
            'OriginNote': {'ReliedNote': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT]}
        }
        
        # FIX: Lambda now accepts 3 arguments to match _get_note_data(name, vault, data)
        mock_get_data.side_effect = lambda name, vault, data: MagicMock() if name in ['OriginNote', 'ReliedNote'] else None
        
        # If your predict_note_linking uses 'cache' as the keyword for existing predictions:
        output = predict_note_linking(
            mock_origin, 
            [mock_relied], 
            MagicMock(),
            cache=mock_cache, # Make sure this matches your function signature
            omit_no_link_predictions=True,
            skip_already_made_predictions=True
        )
        
        # Should return the cached result instead of calling the predictor
        test_eq(output['ReliedNote'], [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT])

In [ ]:
#| hide
def test_prediction_with_missing_label_in_threshold_dict():
    """Test threshold handling when label not in dict."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
        
        # Only specify threshold for one label
        mock_predictor.return_value = [[0.3, 0.8]]
        
        output = prediction_by_note_linking_model(
            mock_origin, 
            mock_relied, 
            mock_predictor, 
            as_floats=False,
            threshold={'INFO_TO_INFO_IN_CONTENT': 0.9}  # NO_LINK not specified, should use default 0.5
        )
        
        test_eq(output['NO_LINK'], False)   # Default 0.5, actual 0.3 < 0.5 = False
        test_eq(output['INFO_TO_INFO_IN_CONTENT'], False)  # Specified threshold 0.9, actual 0.8 < 0.9 = False

test_prediction_with_missing_label_in_threshold_dict()

In [ ]:
#| hide
def test_batch_prediction_with_different_thresholds_per_label():
    """Test batch prediction with different thresholds for each label."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        mock_origin = MagicMock()
        mock_relied = MagicMock()
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
        
        # High probability for INFO_TO_INFO_IN_CONTENT, low for NO_LINK
        mock_predictor.return_value = [
            [0.3, 0.9]  # First pair
        ]
        
        results = batch_prediction(
            [NotePairData(origin_note=mock_origin, relied_note=mock_relied)], 
            mock_predictor, 
            as_floats=False,
            threshold={'NO_LINK': 0.5, 'INFO_TO_INFO_IN_CONTENT': 0.8}
        )
        
        test_eq(results[0]['NO_LINK'], False)   # 0.3 < 0.5 = False
        test_eq(results[0]['INFO_TO_INFO_IN_CONTENT'], True)  # 0.9 > 0.8 = True
test_batch_prediction_with_different_thresholds_per_label()

In [ ]:
def test_batch_prediction_with_very_large_batch():
    """Test batch prediction with large number of pairs."""
    with mock_patch('__main__.string_from_note_pair') as mock_string:
        # Create 100 mock pairs
        origins = [MagicMock() for _ in range(100)]
        relieds = [MagicMock() for _ in range(100)]
        
        mock_predictor = MagicMock()
        mock_predictor.model.config.id2label = {0: 'NO_LINK', 1: 'INFO_TO_INFO_IN_CONTENT'}
        
        # FIX: The mock must return a result for EACH item in the batch provided to it
        def side_effect(batch_texts):
            return [[0.5, 0.5] for _ in batch_texts]
        
        mock_predictor.side_effect = side_effect
        
        pairs = [NotePairData(origin_note=o, relied_note=r) 
                for o, r in zip(origins, relieds)]
        
        # Run with default batch_size (32)
        results = batch_prediction(pairs, mock_predictor, as_floats=True)
        
        # Now it should correctly map all 100
        test_eq(len(results), 100)
        # Verify all results are present and consistent
        test_eq(len(set(r['NO_LINK'] for r in results)), 1)

test_batch_prediction_with_very_large_batch()
# #| hide
# from fastcore.test import *

# test_prediction_with_missing_label_in_threshold_dict()
# test_batch_prediction_with_different_thresholds_per_label()
# test_batch_prediction_with_very_large_batch()